# Preprocessing

Thin wrapper around `data_pipeline.preprocessing.run_preprocessing(...)`.


In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from data_pipeline.preprocessing import resolve_preprocessing_config, run_preprocessing  # noqa: E402

CONFIG = resolve_preprocessing_config()
LANGUAGE_OUTPUTS = {output.name: output for output in CONFIG.language_outputs}
print(f'project_root: {CONFIG.project_root}')
print(f'config_path: {CONFIG.config_path}')
print(f'input_path: {CONFIG.input_path}')
for output_name, output in LANGUAGE_OUTPUTS.items():
    print(f'{output_name}_output_path: {output.output_path}')
print(f'dataset_results_dir: {CONFIG.dataset_results_dir}')
for output_name, output in LANGUAGE_OUTPUTS.items():
    if output.dataset_results_dir is not None:
        print(f'{output_name}_dataset_results_dir: {output.dataset_results_dir}')


project_root: /Users/arikraut/author_attribution/forprosjekt
config_path: /Users/arikraut/author_attribution/forprosjekt/data_pipeline/configs/preprocessing/default.toml
input_path: /Users/arikraut/author_attribution/forprosjekt/data/NPD_v1.csv
bokmal_only_output_path: /Users/arikraut/author_attribution/forprosjekt/data/clean/stortinget_bokmal_only.csv
nynorsk_only_output_path: /Users/arikraut/author_attribution/forprosjekt/data/clean/stortinget_nynorsk_only.csv
dataset_results_dir: /Users/arikraut/author_attribution/forprosjekt/results/dataset
bokmal_only_dataset_results_dir: /Users/arikraut/author_attribution/forprosjekt/results/dataset_bokmal


In [2]:
result = run_preprocessing(CONFIG)
stage_summary_df = result['stage_summary_df']
stage_detail_tables = result['stage_detail_tables']
output_report_df = result['output_report_df']
tables = result['dataset_tables']
bokmal_tables = result['language_dataset_tables']['bokmal_only']


loaded raw data | columns=26
clean dates | rows_removed=0 | authors_removed=0 | invalid_dates=0
keep elections from 2001 onward | rows_removed=590,144 | authors_removed=1,581 | min_election_year=2,001 | rows_before_min_election_year=590,144 | unparseable_election_rows=0
drop procedural speeches | rows_removed=132,478 | authors_removed=3 | excluded_status_rows=121,415 | procedural_column_rows=11,068 | non_status_procedural_column_rows=11,063
apply manual birthyear fixes | birthyear_corrections=56
add age feature | missing_birthyear_before=2 | birthyear_imputed=0 | missing_age_after=2
prepare name redaction values | name_candidates=2,919
select final columns | columns_kept=12 | columns_dropped=15
fill missing party names | missing_partyname_before=21,570 | filled_partyname_rows=21,563 | missing_partyname_after=7
apply manual party fixes | party_codes_normalized=65,874
drop minority-party rows for switchers | rows_removed=3 | party_switcher_authors=1 | party_switcher_rows_dropped=3
drop i

/Users/arikraut/author_attribution/forprosjekt/data_pipeline/preprocessing.py:1225: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stage_summary_df = stage_summary_df.astype(object).fillna("")


In [3]:
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

def display_no_index(df):
    try:
        display(df.style.hide(axis='index'))
    except Exception:
        print(df.to_string(index=False))

print('=== Stage summary ===')
display_no_index(stage_summary_df)
for step, detail_df in stage_detail_tables.items():
    print(f'\n=== {step} ===')
    display_no_index(detail_df)
print('\n=== Output files ===')
display_no_index(output_report_df)
print()
print('=== Corpus overview ===')
display_no_index(tables['corpus_overview'])
print('\n=== Speeches and length per election ===')
display_no_index(tables['election_stats'])
print('\n=== Speech length percentiles (word count) ===')
display_no_index(tables['speech_length_pcts'].T)


=== Stage summary ===


step,rows_after,rows_removed,authors_after,authors_removed
clean dates,982333,0,2618,0
keep elections from 2001 onward,392189,590144,1037,1581
drop procedural speeches,259711,132478,1034,3
drop minority-party rows for switchers,259708,3,,
drop incomplete profiles,259699,9,1032,2
drop duplicate speeches,259672,27,1032,0
keep speeches with at least 150 words,169003,90669,1014,18
keep supported language rows,169003,0,1014,0
filter bokmal_only by small-party threshold,145203,23800,893,121
filter nynorsk_only by small-party threshold,20034,148969,173,841



=== loaded raw data ===


metric,value
columns,26



=== clean dates ===


...

n_speeches,n_authors,n_parties,n_elections,election_range,total_words,total_chars,mean_words_per_speech,median_words_per_speech,female_author_pct
165237,987,7,6,2001–2021,65113630,388107281,394.100000,321.000000,45.100000



=== Speeches and length per election ===


election,n_speeches,n_authors,n_parties,total_words,mean_words,median_words,female_pct,bokmal_pct,nynorsk_pct
2001,26347,255,7,10242224,388.700000,279.000000,37.300000,87.300000,12.700000
2005,26762,301,7,11305790,422.500000,320.000000,43.200000,88.100000,11.900000
2009,29246,288,7,12779088,437.000000,372.000000,41.700000,90.000000,10.000000
2013,28855,304,7,11633379,403.200000,334.000000,44.400000,88.800000,11.200000
2017,33205,311,7,12067542,363.400000,324.000000,46.900000,87.000000,13.000000
2021,20822,273,7,7085607,340.300000,301.000000,45.100000,85.300000,14.700000



=== Speech length percentiles (word count) ===


0,1,2,3,4,5,6,7,8
1,5,10,25,50,75,90,95,99
151,155,161,182,321,488,716,844,1484
